# Exercise 4: storyboard → clips → measure consistency

## Tasks

1. **Plan:** let the OpenAI API draft a short storyboard.
2. **Generate:** create one clip in Google Flow with the character sheet as reference.
3. **Check:** compare the character in the clip with the sheet.
4. **Extension:** more scenes, joined into one film with ffmpeg.

Required: 1 video. No Flow video available? Use Ueli's clip from the course data.

## Setup

Select the **Python 3.11** kernel. The OpenAI API key comes from your `OPENAI_API_KEY` Codespaces secret.

This notebook uses the **OpenAI API** (model `gpt-6-luna`). Other providers, e.g. Google Gemini or Anthropic Claude, have their own APIs with different code.

In [ ]:
# Standard Python libraries.
import os
import sys
from pathlib import Path
from typing import Literal

# Libraries installed from requirements.txt.
import pandas as pd
from PIL import Image
from pydantic import BaseModel

# Work inside this exercise folder, wherever Jupyter starts.
if not Path("data").exists():  # we are inside an exercise folder
    os.chdir("..")  # go to the main folder
os.chdir("04_Storyboard_Consistency")
sys.path.append("..")  # helper.py is in the main folder

# Shared helpers from helper.py.
from helper import DATA, ask, extract_frames

`ask()` from `helper.py` sends a prompt and images to the OpenAI API, writes a log line to `api_log.jsonl` and returns the answer as a Python dict.

## 1. Let the OpenAI API draft a short storyboard

Plan the scenes first, prompt each scene second. The character descriptions stay **exactly the same** in every prompt.

In [ ]:
# Fixed descriptions, used in every prompt.
CHARACTER = "Ueli, red beanie, olive jacket"
DOG = "Bruno, Bernese, red collar"


# One scene of the storyboard.
class Scene(BaseModel):
    id: int
    shot: str
    flow_prompt: str


# The whole storyboard: a list of scenes.
class Storyboard(BaseModel):
    scenes: list[Scene]

In [ ]:
# The instruction for the model.
PROMPT = (
    "Write a storyboard with four short scenes for Google Flow, numbered 1 to 4: "
    "1 a wide mountain lake without characters, 2 a drifting boat, "
    f"3 {CHARACTER} together with the dog {DOG}, 4 sunset. "
    "For shot, give the shot size, e.g. wide or close-up. "
    "Each flow_prompt describes camera, action, light and sound."
)

# Ask for the plan (no images needed) and show it as a table.
plan = ask(PROMPT, Storyboard, [])
pd.DataFrame(plan["scenes"])

In [ ]:
# The Flow prompt for scene 3, with the fixed descriptions in front.
scene_3 = plan["scenes"][2]["flow_prompt"]
print(f"{CHARACTER}. {DOG}. {scene_3}")

## 2. Create one clip in Flow with the character sheet

Use **Veo 3.1 Lite**, 8 seconds and **Ingredients to Video**. Add `ueli_sheet.jpg` as ingredient and paste the prompt above. Upload the clip as `my_scene.mp4` next to this notebook.

In [ ]:
# Show the character sheet.
sheet = DATA / "reference" / "ueli_sheet.jpg"
display(Image.open(sheet).reduce(4))  # a quarter of its size

# Use your own clip, or the course clip if there is none.
video_path = Path("my_scene.mp4")
if not video_path.exists():
    video_path = DATA / "videos" / "ueli_scene3_talk.mp4"
print("Using", video_path.name)

## 3. Compare the character in the clip with the sheet

One frame per second is compared with the character sheet.

In [ ]:
# Save one frame per second of the clip.
paths = extract_frames(video_path)
print(len(paths), "frames saved in frames/")

In [ ]:
# The three allowed ratings.
Rating = Literal["match", "differs", "not visible"]


# The answer format: one rating per feature.
class Consistency(BaseModel):
    beanie: Rating
    jacket: Rating
    dog: Rating


# The instruction: compare image 2 with the reference image 1.
PROMPT = (
    "Image 1 is the reference. Rate the beanie, jacket and dog in image 2. "
    "Use match for a visible match, differs for a visible change, "
    "and not visible when unclear or out of frame."
)

In [ ]:
# Compare every frame with the sheet (reference first).
rows = []
for path in paths:
    row = ask(PROMPT, Consistency, [sheet, path])
    row["frame"] = path.name
    rows.append(row)

# One row per frame.
ratings = pd.DataFrame(rows)
display(ratings)

# Share of "match" per feature.
(ratings[["beanie", "jacket", "dog"]] == "match").mean()

## Extension: join more scenes into one film

Create scenes 1, 2 and 4 in Flow and upload them next to this notebook as `scene_1.mp4`, `scene_2.mp4` and `scene_4.mp4`. Scene 3 is the clip you already used. `-c copy` joins the clips without re-encoding – they need the same format, resolution and frame rate.

In [ ]:
# Your clips, in story order. Scene 3 is the clip from above.
clips = ["scene_1.mp4", "scene_2.mp4", video_path.as_posix(), "scene_4.mp4"]

# ffmpeg reads the list of clips from a text file.
with open("scenes.txt", "w") as file:
    for clip in clips:
        file.write(f"file '{clip}'\n")

# Join the clips into one film. -safe 0 allows paths like ../data/...
!ffmpeg -y -f concat -safe 0 -i scenes.txt -c copy film.mp4

**Check yourself:** If every frame were generated as a separate image, the video would flicker (details jump from frame to frame). Why – and how do video models avoid it?

### Jupyter notebook --footer info--

In [ ]:
# Print system and version info for this run.
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')